<a href="https://colab.research.google.com/github/aahan-charak24/Deep-Learning-Mastery/blob/main/Codes/1D_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

<h1>Load the 1-d MNIST dataset </h1>

In [6]:
from torchvision import datasets, transforms

In [15]:
#transformation to flatten the image into an array
transform  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)) #flatten image into 1-d array
])

#load the mnist dataset and flatten it into a 1-d array
train_data = datasets.MNIST(root = './data', train = True, download = True, transform = transform)
test_data = datasets.MNIST(root = "/.data", train = False, download = True, transform =  transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 483kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.48MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.72MB/s]
100%|██████████| 9.91M/9.91M [00:00<00:00, 16.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 489kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.39MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.90MB/s]


In [16]:
batch_size = 64
#dataloaders
train_loader = DataLoader(train_data, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_data, batch_size = 64, shuffle = True)

In [21]:
#get the first batch to get train and test size
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

torch.Size([64, 784])
torch.Size([64])


<h1>The network will have 3 hidden layers with 15 channels in all layers </h1>

$L_{\text{out}} = \left\lfloor \frac{L_{\text{in}} + 2 \cdot \text{padding} - \text{kernel_size}}{\text{stride}} \right\rfloor + 1$

MNIST flattened: length = 28×28 = 784
1st conv: kernel=3, stride=2, padding=0 →
𝐿
=
⌊
(
784
−
3
)
/
2
⌋
+
1
=
391
L=⌊(784−3)/2⌋+1=391
2nd conv: same →
𝐿
=
⌊
(
391
−
3
)
/
2
⌋
+
1
=
195
L=⌊(391−3)/2⌋+1=195
3rd conv: same →
𝐿
=
⌊
(
195
−
3
)
/
2
⌋
+
1
=
97
L=⌊(195−3)/2⌋+1=97

So flattened size = out_channels * L = 15 * 97 = 1455 → input to FC layer.

In [28]:
class CNN1D(nn.Module):
  def __init__(self):
    super().__init__()
    #first conv layer
    self.conv_layers = nn.Sequential(
        nn.Conv1d(in_channels = 1, out_channels = 15, kernel_size = 3, stride = 2 , padding = 0),
        nn.ReLU(),
        nn.Conv1d(in_channels = 15, out_channels = 15, kernel_size = 3, stride = 2, padding = 0),
        nn.ReLU(),
        nn.Conv1d(in_channels = 15, out_channels = 15, kernel_size = 3, stride = 2, padding = 0),
        nn.ReLU(),
        nn.Flatten(),
    )

    #fully connected
    self.fc = nn.Linear(15*97, 10)


  #forward pass
  def forward(self, x):
    x = self.conv_layers(x)
    x = self.fc(x)

    return x





In [29]:
model = CNN1D()

print(model)

CNN1D(
  (conv_layers): Sequential(
    (0): Conv1d(1, 15, kernel_size=(3,), stride=(2,))
    (1): ReLU()
    (2): Conv1d(15, 15, kernel_size=(3,), stride=(2,))
    (3): ReLU()
    (4): Conv1d(15, 15, kernel_size=(3,), stride=(2,))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
  )
  (fc): Linear(in_features=1455, out_features=10, bias=True)
)


In [35]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(params = model.parameters(), lr = 0.001)

In [41]:
n_epochs = 20
loss_arr = []


for epoch in range(n_epochs):
  #loader
  loss_b = 0.0
  for x_batch, y_batch in train_loader:
    optimizer.zero_grad()
    x_batch = x_batch.unsqueeze(1)

    #forward pass
    y_pred = model(x_batch)

    #calc loss
    loss = criterion(y_pred, y_batch)
    loss_b += loss.item()

    #backprop
    loss.backward()
    optimizer.step()


  #epoch end
  print(f'Epoch: {epoch} train_loss : {loss_b/len(train_loader)}')
  loss_arr.append(loss_b/len(train_loader))


Epoch: 0 train_loss : 0.10888083309316432
Epoch: 1 train_loss : 0.10251425609373843
Epoch: 2 train_loss : 0.09768610041904281
Epoch: 3 train_loss : 0.09295336325277588
Epoch: 4 train_loss : 0.08878275043200026
Epoch: 5 train_loss : 0.08451849510118778
Epoch: 6 train_loss : 0.08205554124364045
Epoch: 7 train_loss : 0.07875455511328794
Epoch: 8 train_loss : 0.07563980631301923
Epoch: 9 train_loss : 0.07330809603321121
Epoch: 10 train_loss : 0.0709982355103405
Epoch: 11 train_loss : 0.0685561458023686
Epoch: 12 train_loss : 0.0666520284283519
Epoch: 13 train_loss : 0.06454823146472925
Epoch: 14 train_loss : 0.06257572958964719
Epoch: 15 train_loss : 0.060327875124080096
Epoch: 16 train_loss : 0.05852617717659307
Epoch: 17 train_loss : 0.0575440909188769
Epoch: 18 train_loss : 0.055346703497635354
Epoch: 19 train_loss : 0.053726758297308407


In [55]:
model.eval()

test_correct = 0
test_total = 0


with torch.no_grad():
  for x_batch, y_batch in test_loader:
    x_batch = x_batch.unsqueeze(1)
    out = model(x_batch)
    _, pred = torch.max(out, axis = 1)

    test_total += y_batch.size(0)
    test_correct += (pred == y_batch).sum().item()

  test_acc = test_correct / test_total
  print(f'Test accuracy is {test_acc *100 :.4f}%\b')


Test accuracy is 96.6100%
